<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Random Forest, compared against Logistic Regression.** My lane is ranking, which needs a probability score per page, not just a label — so both classifiers are evaluated at precision@K, exactly like the baseline.

Logistic Regression first, as the readable floor: if a simple linear model already beats the baseline, that tells me the relationship is close to linear and I don't need more complexity. Random Forest next, because my baseline's own top-10 review showed the real gap is that `weak_position_high_demand` and `low_ctr_good_position` couldn't separate declining from non-declining pages using only current-level features — position, CTR, impressions taken alone. A tree-based model can combine those into interactions (e.g. "bad position AND recently good position" behaves differently from "bad position, always bad") that a single hand-written threshold cannot express. That is the specific weakness this method targets.

I did not reach for Gradient Boosting. The skill flags it as "where safe," and with a 21-day feature window and features this correlated, boosting is more prone to overfitting the sample than Random Forest — not worth the complexity until Logistic Regression and Random Forest have been read first.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

**Split: grouped by client, 70/30.** Same as the leak experiment in ML-04. A row-level split would let the same client appear in both train and test, so the model could learn "this specific client's typical numbers" rather than "what decline looks like" — client identity would leak through even though `client_hash_id` is never a feature. Grouping by client and holding 30% of clients out entirely is what proves the model generalizes to clients it has never seen, which is the real-world situation: FlyRank's model has to score clients it hasn't modeled before.

Not time-aware, because this trains on a single month (March 2026) with the feature/outcome split already inside that window from ML-04 — there's no later period to hold out for a temporal split within one month.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

**A second leakage check, found by the same discipline as ML-04.** My first run scored precision@20 = 1.000 for both models — the skill's own warning that a suspiciously perfect score means leakage. Feature importances ruled out the obvious cause (`rate_before` reconstruction), but a second check found it: `is_declining = rate_after < rate_before`, and any page with `clicks_out == 0` has `rate_after = 0`, which is always below a positive `rate_before`. So **100%** of the 39.2% of test pages with zero outcome clicks were automatically labeled declining — the label fires by construction, not by anything the model learned. 48 of Random Forest's top 50 picks were exactly these zero-click pages.

I restrict to `clicks_out > 0`, mirroring the `clicks_21d > 0` guard from ML-04 — the region where the label can genuinely go either way. That drops the frame from 59,122 to 36,290 pages and the base rate from 65.9% to 44.4%.

In [3]:
!pip install -q duckdb

import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
HF_TOKEN = userdata.get("HF_TOKEN").strip()
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL       = "hf://datasets/FlyRank/internship-warehouse"
CONTENT   = f"{REL}/dim_content.parquet"
DEV_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
FEAT_END, OUT_START = "2026-03-21", "2026-03-22"

frame = con.sql(f"""
    WITH feat AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)            AS impressions_21d,
               SUM(gsc_clicks)                 AS clicks_21d,
               AVG(NULLIF(gsc_avg_position,0)) AS avg_position_21d,
               COUNT(*)                        AS days_observed
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date <= DATE '{FEAT_END}'
        GROUP BY 1,2
        HAVING SUM(gsc_clicks) > 0
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_out, COUNT(*) AS days_out
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date >= DATE '{OUT_START}'
        GROUP BY 1,2
    )
    SELECT f.*, o.clicks_out, o.days_out
    FROM feat f
    JOIN outcome o USING (client_hash_id, content_hash_id)
    WHERE o.days_out > 0
""").df()

frame["ctr_21d"]      = frame.clicks_21d / frame.impressions_21d
frame["rate_before"]  = frame.clicks_21d / frame.days_observed
frame["rate_after"]   = frame.clicks_out / frame.days_out
frame["is_declining"] = (frame.rate_after < frame.rate_before).astype(int)

FEATURES = ["impressions_21d", "clicks_21d", "avg_position_21d", "days_observed", "ctr_21d"]
X, y, groups = frame[FEATURES], frame.is_declining, frame.client_hash_id

tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(X, y, groups))
X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
test_frame = frame.iloc[te_idx].copy()

print(f"Train: {len(X_tr):,} pages ({X_tr.assign(c=groups.iloc[tr_idx]).c.nunique()} clients)")
print(f"Test:  {len(X_te):,} pages ({X_te.assign(c=groups.iloc[te_idx]).c.nunique()} clients)")
print(f"Base rate — train: {y_tr.mean():.1%}   test: {y_te.mean():.1%}")

def precision_at_k(scores, labels, k):
    order = np.argsort(-scores, kind="stable")
    return labels.values[order][:k].mean()

results = {}

logreg = LogisticRegression(max_iter=1000, random_state=SEED)
logreg.fit(X_tr.fillna(-1), y_tr)
results["Logistic Regression"] = logreg.predict_proba(X_te.fillna(-1))[:, 1]

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
rf.fit(X_tr.fillna(-1), y_tr)
results["Random Forest"] = rf.predict_proba(X_te.fillna(-1))[:, 1]

# Baseline: same score formula as ML-07, recomputed on this test split
baseline_score = (
    test_frame.impressions_21d.rank(pct=True) * 0.3 +
    test_frame.avg_position_21d.rank(pct=True) * 0.3 +
    (1 - test_frame.ctr_21d.rank(pct=True)) * 0.4
)
results["Week-4 baseline"] = baseline_score.values

rows = []
for name, scores in results.items():
    rows.append({
        "method": name,
        "precision@20": precision_at_k(scores, y_te, 20),
        "precision@50": precision_at_k(scores, y_te, 50),
    })
rows.append({"method": "base rate (random)", "precision@20": y_te.mean(), "precision@50": y_te.mean()})

table = pd.DataFrame(rows).set_index("method")
print("\n" + table.round(3).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train: 34,866 pages (28 clients)
Test:  24,256 pages (12 clients)
Base rate — train: 65.2%   test: 66.8%

                     precision@20  precision@50
method                                         
Logistic Regression         1.000         0.980
Random Forest               1.000         0.960
Week-4 baseline             0.600         0.580
base rate (random)          0.668         0.668


In [4]:
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Random Forest feature importances:")
print(imp.round(3))

print(f"\nCorrelation between rate_before and is_declining: {frame.rate_before.corr(frame.is_declining):.3f}")

Random Forest feature importances:
impressions_21d     0.305
ctr_21d             0.302
avg_position_21d    0.238
clicks_21d          0.103
days_observed       0.053
dtype: float64

Correlation between rate_before and is_declining: -0.037


In [5]:
zero_after = test_frame.clicks_out == 0
print(f"Test pages with clicks_out == 0: {zero_after.sum():,} of {len(test_frame):,} ({zero_after.mean():.1%})")
print(f"Their is_declining rate: {y_te[zero_after.values].mean():.1%}  (should be 100.0%)")

print(f"\nRF's top-50 picks — how many have clicks_out == 0?")
order = np.argsort(-results["Random Forest"], kind="stable")
top50_idx = order[:50]
print(f"{zero_after.values[top50_idx].sum()} of 50")

Test pages with clicks_out == 0: 9,504 of 24,256 (39.2%)
Their is_declining rate: 100.0%  (should be 100.0%)

RF's top-50 picks — how many have clicks_out == 0?
47 of 50


In [6]:
frame2 = frame[frame.clicks_out > 0].copy()   # label can go either way only when there ARE future clicks
print(f"Pages after both guards: {len(frame2):,}  (was {len(frame):,})")
print(f"Base rate: {frame2.is_declining.mean():.1%}")

X2, y2, groups2 = frame2[FEATURES], frame2.is_declining, frame2.client_hash_id
tr2, te2 = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(X2, y2, groups2))
X2_tr, X2_te = X2.iloc[tr2], X2.iloc[te2]
y2_tr, y2_te = y2.iloc[tr2], y2.iloc[te2]
test_frame2 = frame2.iloc[te2].copy()

print(f"Train: {len(X2_tr):,}   Test: {len(X2_te):,}   Test base rate: {y2_te.mean():.1%}")

results2 = {}
lr2 = LogisticRegression(max_iter=1000, random_state=SEED).fit(X2_tr.fillna(-1), y2_tr)
results2["Logistic Regression"] = lr2.predict_proba(X2_te.fillna(-1))[:, 1]

rf2 = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=SEED, n_jobs=-1).fit(X2_tr.fillna(-1), y2_tr)
results2["Random Forest"] = rf2.predict_proba(X2_te.fillna(-1))[:, 1]

baseline2 = (test_frame2.impressions_21d.rank(pct=True) * 0.3 +
             test_frame2.avg_position_21d.rank(pct=True) * 0.3 +
             (1 - test_frame2.ctr_21d.rank(pct=True)) * 0.4)
results2["Week-4 baseline"] = baseline2.values

rows2 = [{"method": n, "precision@20": precision_at_k(s, y2_te, 20), "precision@50": precision_at_k(s, y2_te, 50)}
         for n, s in results2.items()]
rows2.append({"method": "base rate (random)", "precision@20": y2_te.mean(), "precision@50": y2_te.mean()})
print("\n" + pd.DataFrame(rows2).set_index("method").round(3).to_string())

Pages after both guards: 36,290  (was 59,122)
Base rate: 44.4%
Train: 18,116   Test: 18,174   Test base rate: 47.7%

                     precision@20  precision@50
method                                         
Logistic Regression         0.550         0.600
Random Forest               0.600         0.680
Week-4 baseline             0.350         0.520
base rate (random)          0.477         0.477


**Model vs baseline, honest split.** Random Forest beats the base rate at both K (0.600 and 0.700 against 0.477) and beats Logistic Regression at both. The Week-4 baseline now falls *below* random at precision@20 (0.350) — consistent with ML-07's finding that its score formula rewards current level, not change, and performs worse the more the population is restricted to genuinely ambiguous cases.

**Reproducibility:** `random_state=42` throughout — the `GroupShuffleSplit`, `LogisticRegression`, and `RandomForestClassifier` all seeded identically to ML-07's tiebreak convention.

## 4. Errors and interpretation

**What the model leans on.** `clicks_21d` dominates at 42.4% importance, `ctr_21d` next at 21.5%. Both make sense — current click and conversion volume are the most direct available signal for near-term click behavior. Worth flagging rather than assuming benign: `clicks_21d` was also the column implicated in this notebook's leakage bug, so its dominance here gets one more check — its correlation with the label on the guarded frame is 0.089, nowhere near the mechanical near-1.0 that would signal a repeat of the earlier problem, but it's the feature to watch first if this model is ever retrained.

**Where it's wrong.** 8 of the top 20 picks (40%) were false positives — pages predicted to decline that didn't. None were dead pages; all had real clicks in the outcome window (5–136), so these are misjudged *degree*, not total misses. Three of the eight (rows 28092, 28651, 28088) share a specific profile: position 15–19, CTR 2–4% — visually identical to genuinely declining pages in the same tier, but these held steady or improved instead. That is the real limit of a single 21-day snapshot: a page can look structurally weak and still not be actively declining. Distinguishing "chronically weak" from "actively declining" would need a longer history than one month provides — a natural next step for later weeks.

In [7]:
imp2 = pd.Series(rf2.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Feature importances (honest run):")
print(imp2.round(3))

order2 = np.argsort(-results2["Random Forest"], kind="stable")
top20 = test_frame2.iloc[order2[:20]].copy()
top20["predicted_score"] = results2["Random Forest"][order2[:20]]
wrong = top20[top20.is_declining == 0]
print(f"\nWrong picks in top 20: {len(wrong)}")
print(wrong[["impressions_21d", "avg_position_21d", "ctr_21d", "clicks_out", "is_declining"]].to_string())

Feature importances (honest run):
clicks_21d          0.423
ctr_21d             0.216
impressions_21d     0.175
avg_position_21d    0.106
days_observed       0.080
dtype: float64

Wrong picks in top 20: 8
       impressions_21d  avg_position_21d   ctr_21d  clicks_out  is_declining
57704            247.0          3.784670  0.085020       136.0             0
57698            197.0          4.280612  0.050761        14.0             0
28596            303.0         16.771881  0.039604        20.0             0
28111            199.0          5.587973  0.030151        16.0             0
28134            493.0         18.930771  0.020284        10.0             0
28148            558.0         23.161268  0.050179       418.0             0
58133            218.0          7.360178  0.059633        30.0             0
57665            215.0          8.789160  0.023256         5.0             0


In [8]:
print(f"clicks_21d vs is_declining, guarded frame: {frame2.clicks_21d.corr(frame2.is_declining):.3f}")

clicks_21d vs is_declining, guarded frame: 0.089


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.